In [64]:
import ee
import geopandas as gpd
import requests
import pandas as pd
from PIL import Image
from io import BytesIO
from pathlib import Path
import matplotlib.pyplot as plt


# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project="acm-research-f26")
print("Google Earth Engine initialized successfully.")

Google Earth Engine initialized successfully.


In [65]:
#load data
gdf = pd.read_csv(
    "./Data/im3_atlas_cleaned.csv",
    dtype={"id": str}
)

gdf = gdf[gdf["type"].isin(["building", "campus", "point"])]

# ensure coordinates are numeric
gdf["lat"] = pd.to_numeric(gdf["lat"], errors="coerce")
gdf["lon"] = pd.to_numeric(gdf["lon"], errors="coerce")

gdf = gdf.dropna(subset=["lat", "lon"])

print("Total records:", len(gdf))
print(gdf["type"].value_counts())

Total records: 1147
type
building    1039
campus       108
Name: count, dtype: int64


In [66]:
def get_latest_naip_image(region, start_year=2026, earliest_year=2020):
    for year in range(start_year, earliest_year - 1, -1):
        naip = (
            ee.ImageCollection("USDA/NAIP/DOQQ")
            .filterBounds(region)
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
            .sort("system:time_start", False)

        )
        image_count = naip.size().getInfo()
        print(f"Year: {year}, Image Count: {image_count}")

        if image_count > 0:
            newest = ee.Image(naip.first())
            date = (
                ee.Date(newest.get("system:time_start"))
                .format("YYYY-MM-dd")
                .getInfo()
            )
            image = naip.mosaic()
            return image, date, year
    return None, None, None

In [67]:
#checkpoint download
from huggingface_hub import hf_hub_download

model_name = "ViT-B-32"

checkpoint_path = hf_hub_download(
    repo_id="chendelong/RemoteCLIP",
    filename=f"RemoteCLIP-{model_name}.pt",
    cache_dir="checkpoints"
)

print("Checkpoint:", checkpoint_path)

Checkpoint: /Users/kevinlam/ATLAS/checkpoints/models--chendelong--RemoteCLIP/snapshots/bf1d8a3ccf2ddbf7c875705e46373bfe542bce38/RemoteCLIP-ViT-B-32.pt


In [68]:
#load RemoteCLIP model
import torch
import open_clip

if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using:", device)

model, _, preprocess = open_clip.create_model_and_transforms(
    model_name
)

tokenizer = open_clip.get_tokenizer(model_name)

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu"
)

message = model.load_state_dict(ckpt)
print(message)

model = model.to(device).eval()

Using: mps


<All keys matched successfully>


In [69]:
#NAIP --> RemoteCLIP

valid_queries = [
    "a close overhead aerial photograph centered on one completed data center building with its entire roof visible",
    "a high resolution aerial photograph of a completed data center facility with clearly defined building boundaries",
    "an overhead aerial photograph of a large completed industrial building occupying most of the image",
    "a close aerial view of a completed data center campus with large rectangular buildings and visible cooling equipment",
    "an aerial photograph of a completed industrial facility with its roof, surrounding access roads, and infrastructure clearly visible",
]

invalid_queries = [
    "a wide aerial photograph of a dense urban area with highways, office buildings, and parking lots",
    "an aerial photograph of a residential neighborhood containing many small houses and local streets",
    "an overhead aerial photograph of a city block containing many unrelated buildings with no single dominant facility",
    "an aerial photograph of a commercial district with shopping centers, parking lots, and busy roads",
    "an aerial photograph of an industrial area where the target facility is too small to distinguish",
    "an aerial photograph of a construction site containing unfinished buildings, exposed foundations, and bare soil",
    "an overhead aerial photograph of an incomplete industrial building with an unfinished roof",
]



In [70]:
import numpy as np
import pandas as pd
import torch
import boto3

from pathlib import Path

# AWS S3 configuration
session = boto3.Session(profile_name="ATLAS")

s3 = session.client(
    "s3",
    region_name="us-east-2"
)

BUCKET = "atlas-acm-research-f26"
S3_PREFIX = "satellite/raw/"

# Configuration
NUM_IMAGES = 100

temp_folder = Path("./Data/temp")
Rtemp_folder = Path("./Data/rejectedTemp")
Rtemp_folder.mkdir(parents=True, exist_ok=True)

temp_folder.mkdir(parents=True, exist_ok=True)

# select unique physical locations and not just unique IDs

MIN_DISTANCE_METERS = 3000  # data centers must be at least 3 km apart

#existing ids in S3
existing_ids = set()

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=BUCKET, Prefix=S3_PREFIX):
    for obj in page.get("Contents", []):
        filename = obj["Key"].removeprefix(S3_PREFIX)

        if filename.endswith(".png") and "/" not in filename:
            existing_ids.add(Path(filename).stem)

# coordinates of already-uploaded centers
existing_centers = gdf[
    gdf["id"].astype(str).isin(existing_ids)
]

# candidate centers, excluding uploaded IDs
candidates = (
    gdf
    .drop_duplicates(subset=["id"])
    .loc[lambda df: ~df["id"].isin(existing_ids)]
    .sample(frac=1)  # Random order
)

# distance between geographic coordinates
def distance_meters(lat, lon, other_lats, other_lons):
    R = 6371000

    lat1 = np.radians(lat)
    lon1 = np.radians(lon)
    lat2 = np.radians(np.asarray(other_lats))
    lon2 = np.radians(np.asarray(other_lons))

    a = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin((lon2 - lon1) / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


# start with locations already represented in S3
used_lats = existing_centers["lat"].tolist()
used_lons = existing_centers["lon"].tolist()

selected_indices = []

for idx, row in candidates.iterrows():
    lat = float(row["lat"])
    lon = float(row["lon"])

    if used_lats:
        distances = distance_meters(lat, lon, used_lats, used_lons)

        # skip the same or nearby facility
        if np.any(distances < MIN_DISTANCE_METERS):
            continue

    selected_indices.append(idx)
    used_lats.append(lat)
    used_lons.append(lon)

    if len(selected_indices) >= NUM_IMAGES:
        break

data_centers = candidates.loc[selected_indices]

print("Already in S3:", len(existing_ids))
print("Unique locations selected:", len(data_centers))

# encode RemoteCLIP text descriptions once
text_queries = valid_queries + invalid_queries
text_input = tokenizer(text_queries).to(device)

with torch.no_grad():
    text_features = model.encode_text(text_input)
    text_features /= text_features.norm(dim=-1, keepdim=True)

results = []

for count, (_, data_center) in enumerate(data_centers.iterrows(), start=1):

    center_id = str(data_center["id"])

    print(f"\n[{count}/{len(data_centers)}] {center_id}")

    result = {
        "id": center_id,
        "date": None,
        "decision": "REJECT",
        "reason": None,
        "margin": None,
        "image_path": None,
        "s3_key": None,
        "upload_status": "NOT_UPLOADED"
    }

    try:
        # get coordinates and buffer size
        lon = float(data_center["lon"])
        lat = float(data_center["lat"])

        buffer_meters = 300

        building_ee = ee.Geometry.Point([lon, lat])
        region = building_ee.buffer(buffer_meters).bounds()

        #retrieve NAIP imagery
        image, date, year = get_latest_naip_image(region)

        if image is None:
            result["reason"] = "NO_NAIP_IMAGE"
            print("REJECT: No NAIP imagery")
            results.append(result)
            continue

        result["date"] = date

        #same dimensions for all images
        IMAGE_SIZE = 1000  

        rgb = image.select(["R", "G", "B"])

        url = rgb.getThumbURL({
            "bands": ["R", "G", "B"],
            "min": 0,
            "max": 255,
            "dimensions": f"{IMAGE_SIZE}x{IMAGE_SIZE}",
            "region": region,
            "format": "png"
        })

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        satellite_image = Image.open(
            BytesIO(response.content)
        ).convert("RGB")


        if satellite_image.size != (IMAGE_SIZE, IMAGE_SIZE):
            satellite_image = satellite_image.resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.LANCZOS
            )

        #check for blank or invalid imagery
        pixels = np.asarray(satellite_image)

        if (
            pixels.size == 0
            or pixels.std() < 5
            or pixels.mean() < 5
            or pixels.mean() > 250
        ):
            result["reason"] = "INVALID_IMAGE"
            print("REJECT: Invalid image")
            results.append(result)
            continue

        # encode image using RemoteCLIP
        image_input = (
            preprocess(satellite_image)
            .unsqueeze(0)
            .to(device)
        )

        with torch.no_grad():
            image_features = model.encode_image(image_input)

            image_features /= image_features.norm(
                dim=-1,
                keepdim=True
            )

            similarities = (
                image_features @ text_features.T
            )[0]


        num_valid = len(valid_queries)

        best_valid = similarities[:num_valid].max().item()
        best_invalid = similarities[num_valid:].max().item()

        margin = best_valid - best_invalid
        result["margin"] = margin

        if margin > 0:
            decision = "KEEP"
        else:
            decision = "REJECT"

        result["decision"] = decision
        result["reason"] = "REMOTECLIP_CLASSIFICATION"

        file_path = temp_folder / f"{center_id}.png"
        Rfile_path = Rtemp_folder / f"{center_id}.png"

        if decision == "KEEP":

            #saved locally for manual review
            satellite_image.save(file_path)
            result["image_path"] = str(file_path)

            # upload accepted image to S3
            s3_key = f"{S3_PREFIX}{center_id}.png"

            try:
                s3.upload_file(
                    str(file_path),
                    BUCKET,
                    s3_key,
                    ExtraArgs={"ContentType": "image/png"}
                )

                result["s3_key"] = s3_key
                result["upload_status"] = "SUCCESS"

                print(f"KEEP | Margin: {margin:.4f}")
                print(f"Uploaded: s3://{BUCKET}/{s3_key}")

            except Exception as upload_error:
                result["upload_status"] = "FAILED"
                result["upload_error"] = str(upload_error)

                print(f"KEEP | S3 upload failed: {upload_error}")

        else:

            if file_path.exists():
                file_path.unlink()

            satellite_image.save(Rfile_path)
            result["image_path"] = str(Rfile_path)
            result["upload_status"] = "NOT_UPLOADED"

            print(f"REJECT | Margin: {margin:.4f}")

        # record every processed image
        results.append(result)

    except Exception as e:
        result["decision"] = "ERROR"
        result["reason"] = str(e)

        results.append(result)

        print(f"ERROR: {e}")

#save validation results
results_df = pd.DataFrame(results)

results_df.to_csv(
    "./Data/validation_results.csv",
    index=False
)

print("\n--- BATCH SUMMARY ---")
print(results_df["decision"].value_counts())

print(
    "Accepted images:",
    (results_df["decision"] == "KEEP").sum()
)

print(
    "Successfully uploaded:",
    (results_df["upload_status"] == "SUCCESS").sum()
)


Traceback (most recent call last):
  File "/Users/kevinlam/ATLAS/.venv/lib/python3.13/site-packages/botocore/credentials.py", line 2633, in refresh_credentials
    response = client.create_o_auth2_token(
        tokenInput={
    ...<3 lines>...
        },
    )
  File "/Users/kevinlam/ATLAS/.venv/lib/python3.13/site-packages/botocore/client.py", line 606, in _api_call
    return self._make_api_call(operation_name, kwargs)
           ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/kevinlam/ATLAS/.venv/lib/python3.13/site-packages/botocore/context.py", line 123, in wrapper
    return func(*args, **kwargs)
  File "/Users/kevinlam/ATLAS/.venv/lib/python3.13/site-packages/botocore/client.py", line 1094, in _make_api_call
    raise error_class(parsed_response, operation_name)
botocore.errorfactory.AccessDeniedException: An error occurred (AccessDeniedException) when calling the CreateOAuth2Token operation: The refresh token has expired.

The above exception was the direct cause of

LoginRefreshRequired: Your session has expired or credentials have changed. Please reauthenticate using 'aws login'.